<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkST0151ENSkillsNetwork20531532-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>



# **Final Project: Boston Housing**


##### Estimated time needed: **60** minutes


#### Import the required libraries we need for the lab.


In [ ]:
import piplite
await piplite.install(['numpy', 'pandas', 'seaborn', 'statsmodels'])

In [ ]:
import pandas as pd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pyplot
import scipy.stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

#### Read the dataset in the csv file from the URL


In [ ]:
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv'
resp = await fetch(URL)
boston_url = io.BytesIO((await resp.arrayBuffer()).to_py())

In [ ]:
boston_df=pd.read_csv(boston_url)

#### Add your code below following the instructions given in the course to complete the peer graded assignment


## Data preparation

In [ ]:
# The CSV has an extra index column; remove it so it does not affect the correlations
if "Unnamed: 0" in boston_df.columns:
    boston_df = boston_df.drop(columns=["Unnamed: 0"])

# Group AGE into three categories for later exercises
boston_df.loc[boston_df["AGE"] <= 35, "age_group"] = "35 years and younger"
boston_df.loc[(boston_df["AGE"] > 35) & (boston_df["AGE"] < 70), "age_group"] = "between 35 and 70 years"
boston_df.loc[boston_df["AGE"] >= 70, "age_group"] = "70 years and older"
boston_df.head()

## Basic exploration

In [ ]:
# First 5 rows
boston_df.head()

In [ ]:
# Data type of each column
boston_df.dtypes

In [ ]:
# Missing values in each column
boston_df.isnull().sum()

In [ ]:
# Number of duplicate rows
boston_df.duplicated().sum()

In [ ]:
# Correlation matrix (numeric columns only)
boston_df.corr(numeric_only=True)

In [ ]:
# Summary statistics
boston_df.describe()

## Visualizations

### Boxplot: median value of owner-occupied homes (MEDV)

In [ ]:
pyplot.figure(figsize=(6, 5))
ax = sns.boxplot(y="MEDV", data=boston_df)
ax.set_ylabel("Median value of homes ($1000s)")
ax.set_title("Boxplot of MEDV")
pyplot.show()

**Interpretation:** Most median home values fall between roughly 17 and 25 (in $1000s), with a median around 21. There are several high outliers, including a cluster at the top of the range (about 50), so the distribution is skewed toward higher values.

### Histogram: Charles River variable (CHAS)

In [ ]:
pyplot.figure(figsize=(6, 5))
ax = sns.countplot(x="CHAS", data=boston_df)
ax.set_xlabel("Bounded by Charles River (0 = No, 1 = Yes)")
ax.set_ylabel("Number of towns")
ax.set_title("Towns bounded by the Charles River")
pyplot.show()
print(boston_df["CHAS"].value_counts())

**Interpretation:** The large majority of towns are not bounded by the Charles River (CHAS = 0). Only a small number (about 35 of 506) border the river.

### Boxplot: MEDV vs AGE group

In [ ]:
order = ["35 years and younger", "between 35 and 70 years", "70 years and older"]
pyplot.figure(figsize=(8, 5))
ax = sns.boxplot(x="age_group", y="MEDV", data=boston_df, order=order)
ax.set_xlabel("Age group (proportion of owner-occupied units built before 1940)")
ax.set_ylabel("Median value of homes ($1000s)")
ax.set_title("MEDV by age group")
pyplot.show()

**Interpretation:** Median home values decrease as the age group gets older. Towns in the youngest group have the highest median values, and towns in the oldest group (70 years and older) have the lowest.

### Scatter plot: nitric oxide (NOX) vs non-retail business acres (INDUS)

In [ ]:
pyplot.figure(figsize=(7, 5))
ax = sns.scatterplot(x="INDUS", y="NOX", data=boston_df)
ax.set_xlabel("Proportion of non-retail business acres per town (INDUS)")
ax.set_ylabel("Nitric oxide concentration (NOX)")
ax.set_title("NOX vs INDUS")
pyplot.show()

**Interpretation:** There is a clear positive relationship. Towns with a higher proportion of non-retail (industrial) land tend to have higher nitric oxide concentrations.

### Histogram: pupil-to-teacher ratio (PTRATIO)

In [ ]:
pyplot.figure(figsize=(7, 5))
ax = sns.histplot(boston_df["PTRATIO"], bins=15)
ax.set_xlabel("Pupil-to-teacher ratio (PTRATIO)")
ax.set_ylabel("Number of towns")
ax.set_title("Distribution of PTRATIO")
pyplot.show()

**Interpretation:** PTRATIO is not symmetric. Most towns have a ratio between about 14 and 21, with the largest group at the higher end (around 20 to 21) and a smaller tail toward lower ratios.

## Hypothesis tests

### 1. Is there a significant difference in median value of houses bounded by the Charles River or not? (t-test)

**H0:** The mean MEDV is the same for houses bounded and not bounded by the Charles River.  
**H1:** The mean MEDV is different.  
**Significance level:** 0.05

In [ ]:
river = boston_df[boston_df["CHAS"] == 1]["MEDV"]
no_river = boston_df[boston_df["CHAS"] == 0]["MEDV"]

# Check equality of variances first
lev = scipy.stats.levene(river, no_river, center="mean")
print("Levene's test p-value:", lev.pvalue)

# t-test; if variances differ (Levene p < 0.05) use Welch's t-test
t_stat, p_val = scipy.stats.ttest_ind(river, no_river, equal_var=(lev.pvalue >= 0.05))
print("t-statistic:", t_stat)
print("p-value:", p_val)

**Conclusion:** The p-value is below 0.05, so we reject the null hypothesis. There is a statistically significant difference in median home value between houses bounded by the Charles River and those that are not. Houses bounded by the river have higher median values.

### 2. Is there a difference in median values of houses for each proportion of owner-occupied units built before 1940? (ANOVA)

**H0:** The mean MEDV is equal across all three age groups.  
**H1:** At least one age group has a different mean MEDV.  
**Significance level:** 0.05

In [ ]:
g1 = boston_df[boston_df["age_group"] == "35 years and younger"]["MEDV"]
g2 = boston_df[boston_df["age_group"] == "between 35 and 70 years"]["MEDV"]
g3 = boston_df[boston_df["age_group"] == "70 years and older"]["MEDV"]

print("Levene's test p-value:", scipy.stats.levene(g1, g2, g3, center="mean").pvalue)

f_stat, p_val = scipy.stats.f_oneway(g1, g2, g3)
print("F-statistic:", f_stat)
print("p-value:", p_val)

**Conclusion:** The p-value is far below 0.05, so we reject the null hypothesis. Median home values differ significantly between at least two of the age groups: towns with older housing tend to have lower median values.

### 3. Can we conclude there is a relationship between nitric oxide concentrations and the proportion of non-retail business acres per town? (Pearson correlation)

**H0:** There is no relationship between NOX and INDUS.  
**H1:** There is a relationship between NOX and INDUS.  
**Significance level:** 0.05

In [ ]:
r, p_val = scipy.stats.pearsonr(boston_df["NOX"], boston_df["INDUS"])
print("Pearson correlation coefficient:", r)
print("p-value:", p_val)

**Conclusion:** The p-value is far below 0.05, so we reject the null hypothesis. There is a statistically significant and strong positive relationship (r is about 0.76) between nitric oxide concentration and the proportion of non-retail business acres per town.

### 4. What is the impact of an additional weighted distance to the five Boston employment centres on the median value of owner-occupied homes? (Regression)

**H0:** The coefficient of DIS is 0 (no effect on MEDV).  
**H1:** The coefficient of DIS is not 0.  
**Significance level:** 0.05

In [ ]:
X = boston_df["DIS"]
y = boston_df["MEDV"]
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
pyplot.figure(figsize=(7, 5))
ax = sns.regplot(x="DIS", y="MEDV", data=boston_df, line_kws={"color": "red"})
ax.set_xlabel("Weighted distance to five Boston employment centres (DIS)")
ax.set_ylabel("Median value of homes ($1000s)")
ax.set_title("MEDV vs DIS")
pyplot.show()

**Conclusion:** The coefficient for DIS is positive (about 1.09) and its p-value is below 0.05, so we reject the null hypothesis. Each additional unit of weighted distance to the employment centres is associated with an increase of about 1.09 (roughly $1,090) in median home value. However, R-squared is low (about 0.06), so distance alone explains only a small part of the variation in home values.

## Summary for management

1. Houses bounded by the Charles River have significantly higher median values.
2. Median values fall as the share of older (pre-1940) homes rises.
3. Nitric oxide concentration rises strongly with the share of non-retail business acres.
4. Greater distance from employment centres is linked to slightly higher home values, but distance alone is a weak predictor.